
# Reti neurali ricorrenti (RNN)

Librerie necessarie:
- `pytorch` 
    - https://pytorch.org/
    - https://docs.pytorch.org/docs/stable/index.html
- Utilità: `os`, `glob`

#### Caricamento del dataset

In [217]:
import torch
import torch.nn as nn
import torch.optim as optim

import os
import glob
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Definzione dell'embedding, manuale con one-hot labels

In [218]:
import string
import unicodedata

ALLOWED_CHARACTERS = string.ascii_letters + " .,;'" + "_"

N_LETTERS = len(ALLOWED_CHARACTERS)

# Conversione caratteri
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in ALLOWED_CHARACTERS
    )

print(f"Name 'Nicolò' becomes {unicode_to_ascii('Nicolò')}")
print(f"Name 'Francesco Maria' becomes {unicode_to_ascii('Francesco Maria')}")

Name 'Nicolò' becomes Nicolo
Name 'Francesco Maria' becomes Francesco Maria


In [219]:
# Tokenizzazione
def letter_to_index(letter):
    if letter not in ALLOWED_CHARACTERS:
        return ALLOWED_CHARACTERS.find("_")
    else:
        return ALLOWED_CHARACTERS.find(letter)

# One-hot encoding --- Embedding!
def line_to_tensor(line):
    tensor = torch.zeros(len(line), 1, N_LETTERS)
    for idx, letter in enumerate(line):
        tensor[idx][0][letter_to_index(letter)] = 1  #ritorna 1 se il carattere è presente, altrimenti 0
    return tensor

print(f"Name 'Nicolò' becomes {line_to_tensor('Nicolò')}")
print(f"Name 'Francesco Maria' becomes {line_to_tensor('Francesco Maria')}")

Name 'Nicolò' becomes tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0

Un altro modo di fare l'embedding:

In [220]:
def embedding(text, vocab_size, embedding_dim):
    #------
    embed = nn.Embedding(vocab_size, embedding_dim)
    #------

    char_indices = [ord(c) for c in text]

    text_tensor = torch.tensor(char_indices, dtype=torch.long)

    embedded_text = embed(text_tensor)

    return embedded_text

Creazione personalizzata del dataset

In [221]:
class NamesDataset(Dataset):
    def __init__(self, folder_path):
        self.data = []
        self.labels = []
        self.labels_uniq = set()
        
        self.data_tensor = []
        self.labels_tensors = []

        # Read all .txt files in the folder
        for file_path in glob.glob(os.path.join(folder_path, "*.txt")):
            label = os.path.splitext(os.path.basename(file_path))[0]
            self.labels_uniq.add(label)
            with open(file_path, "r", encoding="utf-8") as file:
                for line in file:
                    self.data.append(line.strip())
                    #------
                    self.data_tensor.append(line_to_tensor(line.strip()).transpose(0, 1).squeeze(0))  # Convert to (seq_len, embedding_dim)
                    #------
                    self.labels.append(label)

        # Map labels to indices
        self.label_to_idx = {label: idx for idx, label in enumerate(self.labels_uniq)}

        self.labels_uniq = list(self.labels_uniq)
        for label in self.labels:
            #-------
            label_tensor = torch.tensor(self.label_to_idx[label], dtype=torch.long)
            self.labels_tensors.append(label_tensor)
            #-------


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]
        label = self.labels[idx]
        # text_tensor = line_to_tensor(text.strip())
        text_tensor = self.data_tensor[idx]
        # label_tensor = torch.tensor(self.label_to_idx[label], dtype=torch.long)
        label_tensor = self.labels_tensors[idx]
        return label_tensor, text_tensor, label, text

# Initialize the dataset
dataset = NamesDataset("data/names")
print(f"There are {len(dataset)} names in the dataset.")
print(f"Labels are {dataset.labels_uniq}")
print(f"For example, {dataset[0][3]} is a {dataset[0][2]} name,")
print(f"{dataset[0][1]} is the embedding and {dataset[0][0]} the label idx.")

There are 20074 names in the dataset.
Labels are ['Czech', 'German', 'French', 'Russian', 'Greek', 'Irish', 'Portuguese', 'Italian', 'Chinese', 'Polish', 'Scottish', 'Japanese', 'Spanish', 'Arabic', 'Dutch', 'Vietnamese', 'Korean', 'English']
For example, Aalsburg is a Dutch name,
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0

In [222]:
# Create the DataLoader
train_set, test_set = torch.utils.data.random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))])

batch_size = 32

def collate_fn(batch):
    labels, text_tensors, label_names, texts = zip(*batch)
    return (
        torch.stack(labels),
        torch.nn.utils.rnn.pad_sequence(text_tensors, batch_first=True),
        label_names,
        texts,
    )

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"train examples = {len(train_set)}, validation examples = {len(test_set)}")

train examples = 16059, validation examples = 4015


#### Configurazione degli iperparametri

In [223]:
# Hyperparameters
num_epochs = 50
learning_rate = 0.05
torch.manual_seed(42)

#### Definizione della rete neurale

Creare un modulo pytorch (classe derivata da `nn.Module`) che definisce l'architettura della rete.  
Dev'essere presente uno strato ricorrente e un layer FC seguito da una funzione di attivazione log-softmax.

In [224]:
# Define the RNN model
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes) # layer fully connected per la classificazione
        self.log_softmax = nn.LogSoftmax(dim=1) # funzione di attivazione log softmax per la classificazioneòp

    def forward(self, x):
        _, hidden = self.rnn(x)  # out: (batch_size, seq_length, hidden_size)
        out = self.fc(hidden[0])  # Prendi l'output dell'ultimo
        out = self.log_softmax(out)  # Apply log softmax
        return out

# Initialize the model, loss function, and optimizer
input_size = N_LETTERS # Size of the embedding
hidden_size = 128  # Size of the hidden state
num_classes = len(dataset.labels_uniq)  # Number of unique labels
model = SimpleRNN(input_size, hidden_size, num_classes).to(device)

print(model)

SimpleRNN(
  (rnn): RNN(58, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=18, bias=True)
  (log_softmax): LogSoftmax(dim=1)
)


In [225]:
input = line_to_tensor('Nicolò').to(device)

output = model(input)

# print(f"Output for the input 'Nicolò' is: {output}")
_, predicted_label_idx = output.topk(1)
predicted_label_idx = predicted_label_idx[0]

predicted_label = list(dataset.label_to_idx.keys())[list(dataset.label_to_idx.values()).index(predicted_label_idx.item())]
print(f"The predicted label for the input 'Nicolò' is: {predicted_label}")

The predicted label for the input 'Nicolò' is: Italian


#### Scelta della funzione Loss e dell'ottimizzatore

- Cross-entropy loss
- SGD optimizer

In [226]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

Implementare l'algoritmo di training della rete neurale sopra definita, ricordando che:
- per un certo numero di epoche, si itera su tutto il dataset
    - inoltre, i comandi `inputs.to(device), labels.to(device)` assicurano di utilizzare la GPU
- l'ottimizzatore va resettato a ogni iterazione, con `optimizer.zero_grad()`
- per ogni iterazione, si procede con il passo forward
- per ogni iterazione, si calcola la loss relativa a quell'output della rete
    - con `loss = criterion(outputs, labels)`
- per ogni iterazione, si propaga all'indietro la loss
- per ogni iterazione, si esegue uno step dell'ottimizzatore

In [227]:
# Training the model
model.train()
for epoch in range(num_epochs):
    for i, (labels, text_tensor, label, text) in enumerate(train_loader):
        # -------
        labels = labels.to(device)
        text_tensor = text_tensor.to(device)

        outputs = model(text_tensor)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
       

        if (i + 1) % 100 == 0:
            print(f"Batch {i + 1}/{len(train_loader)}: labels shape = {labels.shape}, text_tensor shape = {text_tensor.shape}")
            print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {loss.item():.4f}')


Batch 100/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 12, 58])
Epoch [1/50], Step [100/502], Loss: 1.6137
Batch 200/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 10, 58])
Epoch [1/50], Step [200/502], Loss: 1.7090
Batch 300/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 12, 58])
Epoch [1/50], Step [300/502], Loss: 1.6946
Batch 400/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 12, 58])
Epoch [1/50], Step [400/502], Loss: 1.7356
Batch 500/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 13, 58])
Epoch [1/50], Step [500/502], Loss: 1.5489
Batch 100/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 10, 58])
Epoch [2/50], Step [100/502], Loss: 1.6494
Batch 200/502: labels shape = torch.Size([32]), text_tensor shape = torch.Size([32, 14, 58])
Epoch [2/50], Step [200/502], Loss: 1.9044
Batch 300/502: labels shape = torch.Size([32]), 

#### Inferenza

In [228]:
# Make inference on the first 64 instances of the test set
model.eval()
with torch.no_grad():
    for i, (labels, text_tensor, label_names, texts) in enumerate(test_loader):
        if i == 0:  # Only process the first batch
            text_tensor = text_tensor.to(device)
            outputs = model(text_tensor)
            _, predicted = torch.max(outputs, 1)
            predicted_labels = [list(dataset.label_to_idx.keys())[list(dataset.label_to_idx.values()).index(idx.item())] for idx in predicted]
            index = 0
            for text, predicted_label in zip(texts, predicted_labels):
                print(f"Text: {text}, Predicted Label: {predicted_label}, Real Label: {label_names[index]}")
                index += 1
            break

    # Calcolo dell'accuracy sul test set
    correct = 0
    total = 0
    for labels, text_tensor, label_names, texts in test_loader:
        text_tensor = text_tensor.to(device)
        outputs = model(text_tensor)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted.cpu() == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Accuracy on test set: {accuracy:.2f}%")



Text: Portelli, Predicted Label: Italian, Real Label: Italian
Text: Craven, Predicted Label: English, Real Label: English
Text: Novoshinsky, Predicted Label: Russian, Real Label: Russian
Text: Eddleston, Predicted Label: English, Real Label: English
Text: Rocchi, Predicted Label: Italian, Real Label: Italian
Text: Mcrae, Predicted Label: English, Real Label: English
Text: Mkervali, Predicted Label: Russian, Real Label: Russian
Text: Barron, Predicted Label: English, Real Label: English
Text: Moreno, Predicted Label: Italian, Real Label: English
Text: Voakes, Predicted Label: Dutch, Real Label: English
Text: Khoury, Predicted Label: Arabic, Real Label: Arabic
Text: Watnabe, Predicted Label: English, Real Label: Japanese
Text: Chevrolet, Predicted Label: English, Real Label: French
Text: Luhvich, Predicted Label: Russian, Real Label: Russian
Text: Kamon, Predicted Label: Russian, Real Label: Japanese
Text: Pohodeev, Predicted Label: Russian, Real Label: Russian
Text: Limoshin, Predicted 

### Fine-tuning degli iperparametri

La rete non è ottimale. Cosa fare per migliorare il funzionamento?

In [ ]:
# 20 epoche, 0.01 lr, accuracy 33%, batch size 64
# 20 epoche, 0.05 lr, accuracy 67%, batch size 32
# 40 epoche, 0.05 lr, accuracy 76%, batch size 32, random seed
# 40 epoche, 0.1 lr, accuracy 68%, batch size 32
# 40 epoche, 0.07 lr, accuracy 57%, batch size 32

# 40 epoche, 0.05 lr, accuracy 74%, batch size 32, seed = 42
# 50 epoche, 0.05 lr, accuracy 76%, batch size 32, seed = 42

# si possono aumentare il numero di layer come altro test
